# EASM scan: wikimedia.org

Triggers a scan through the discovery-scheduler pod, tails the pipeline pod logs, and queries the Postgres gold tables.
Requires `kubectl` pointed at the local kind cluster (namespace `easm`) and `pandas`.

> The deployments currently run with `OFFLINE_MODE=true` (synthetic CT records and probes). For real discovery/probing of wikimedia.org set `LIVE = True` in section 1.

In [4]:
import json, subprocess, time
from io import StringIO
import pandas as pd

NS = "easm"
TENANT = "tenant_wikimedia"
DOMAIN = "wikimedia.org"

def kubectl(*args, check=True):
    r = subprocess.run(["kubectl", "-n", NS, *args], capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(r.stderr or r.stdout)
    return r.stdout

def sql(query):
    out = kubectl("exec", "deploy/postgres", "--", "psql", "-U", "easm", "-d", "easm", "--csv", "-c", query)
    return pd.read_csv(StringIO(out))

def logs(app, since="10m", tail=200, grep=None):
    """Logs from all pods labelled app=<name>, prefixed with the pod name."""
    out = kubectl("logs", "-l", f"app={app}", "--prefix", f"--since={since}", f"--tail={tail}", "--max-log-requests=10", check=False)
    lines = [l for l in out.splitlines() if not grep or grep in l]
    print("\n".join(lines) or f"(no logs for app={app})")

print(kubectl("get", "pods"))

NAME                                      READY   STATUS    RESTARTS      AGE
asset-api-67659b8fd6-jsn8f                1/1     Running   0             101m
discovery-scheduler-6454f9dcf-dr4fx       1/1     Running   0             53m
postgres-79967df69f-bgktx                 1/1     Running   1 (94m ago)   95m
probe-worker-7bbf47445f-8bfrd             1/1     Running   0             53m
probe-worker-7bbf47445f-q2h6z             1/1     Running   0             53m
probe-worker-7bbf47445f-xr245             1/1     Running   0             53m
redis-79cfc78c7-mbwcv                     1/1     Running   0             88m
redpanda-8f59ffbfb-5kw9j                  1/1     Running   1 (95m ago)   95m
spark-stream-processor-66769ddbfd-w5hvj   1/1     Running   0             58m



## 1. (Optional) Enable live mode
Switches scheduler and probe workers to `OFFLINE_MODE=false` (queries crt.sh and probes wikimedia.org hosts over DNS/TLS/HTTP). Patches the running deployments only; `k8s/02-apps.yaml` is unchanged.

In [5]:
LIVE = False
if LIVE:
    for d in ("discovery-scheduler", "probe-worker"):
        kubectl("set", "env", f"deploy/{d}", "OFFLINE_MODE=false")
        kubectl("rollout", "status", f"deploy/{d}", "--timeout=180s")

## 2. Trigger the scan

In [6]:
payload = json.dumps({"tenant_id": TENANT, "root_domain": DOMAIN})
script = (
    "import urllib.request as u;"
    f"r=u.Request('http://localhost:8000/scans',data={payload!r}.encode(),headers={{'Content-Type':'application/json'}});"
    "print(u.urlopen(r,timeout=120).read().decode())"
)
print(kubectl("exec", "deploy/discovery-scheduler", "--", "python", "-c", script))

{"tenant_id":"tenant_wikimedia","root_domain":"wikimedia.org","tasks_enqueued":1}



## 3. Pod logs

In [7]:
print("=== discovery-scheduler ==="); logs("discovery-scheduler", since="5m")
print("\n=== probe-worker (x3) ==="); logs("probe-worker", since="5m")

=== discovery-scheduler ===
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler] INFO:     127.0.0.1:47584 - "POST /scans HTTP/1.1" 500 Internal Server Error
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler] ERROR:    Exception in ASGI application
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler] Traceback (most recent call last):
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler]   File "/usr/local/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler]     yield
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler]   File "/usr/local/lib/python3.12/site-packages/httpx/_transports/default.py", line 394, in handle_async_request
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler]     resp = await self._pool.handle_async_request(req)
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler]            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[pod/discovery-scheduler-6454f9dcf-dr4fx/scheduler]   Fil

In [8]:
print("=== spark-stream-processor ==="); logs("spark-stream-processor", since="5m", tail=100)
print("\n=== asset-api ==="); logs("asset-api", since="5m", tail=50)

=== spark-stream-processor ===
(no logs for app=spark-stream-processor)

=== asset-api ===
(no logs for app=asset-api)


## 4. Wait for results in Postgres
Spark micro-batches sync to Postgres, so this can take a minute or two.

In [9]:
assets = pd.DataFrame()
for _ in range(30):
    assets = sql(f"SELECT * FROM gold_assets WHERE tenant_id = '{TENANT}' ORDER BY fqdn")
    if len(assets):
        break
    time.sleep(10)
print(f"{len(assets)} assets")

143 assets


## 5. Query tables

In [10]:
pd.set_option("display.max_columns", None, "display.width", 250, "display.max_colwidth", 80)
assets

,asset_id,tenant_id,fqdn,cluster_id,root_cause_node,latest_ips_json,is_shared_cdn,detected_tech_json,tls_version,cert_sha256,status,exposure_type,validation_tier,first_seen,last_seen
0,2ab227726557d20b3b12,tenant_wikimedia,alert1001.wikimedia.org,fqdn:alert1001.wikimedia.org,fqdn:alert1001.wikimedia.org,[],f,[],NaN,NaN,clean,NaN,informational,2026-11-28T08:54:08.636927+00:00,2026-11-28T08:54:08.636927+00:00
1,ae8abc5f4ad04eb0294c,tenant_wikimedia,alert2001.wikimedia.org,fqdn:alert2001.wikimedia.org,fqdn:alert2001.wikimedia.org,[],f,[],NaN,NaN,clean,NaN,informational,2026-11-28T08:54:08.785418+00:00,2026-11-28T08:54:08.785418+00:00
2,7c0a8ccf03e76a4801b0,tenant_wikimedia,alerts.wikimedia.org,cert:b62fc571af83d8b08d3788f8bf13d5b035be415e4b0af733df022efe895f19cc,fqdn:alert1002.wikimedia.org,"[""208.80.154.78""]",f,"[""server:envoy""]",TLSv1.3,b62fc571af83d8b08d3788f8bf13d5b035be415e4b0af733df022efe895f19cc,clean,NaN,informational,2026-11-28T08:54:09.060837+00:00,2026-11-28T08:54:11.340115+00:00
3,ec77c0f8ba382d418c11,tenant_wikimedia,analytics.frdev.wikimedia.org,cert:b7c48ae781e5b34f9cd11911afb8ba1ab11905e456abdeb26503d84b06a515a4,fqdn:analytics-eqiad.wikimedia.org,"[""208.80.155.4""]",f,"[""server:nginx""]",TLSv1.3,b7c48ae781e5b34f9cd11911afb8ba1ab11905e456abdeb26503d84b06a515a4,clean,NaN,informational,2026-11-28T08:54:13.357610+00:00,2026-11-28T08:54:13.357610+00:00
4,d30b2449b024bb1f0479,tenant_wikimedia,apt.wikimedia.org,cert:2a315459e463d249065f83e3fdf83c6083add95422a202f5af8bfdad63b9f72a,cert:2a315459e463d249065f83e3fdf83c6083add95422a202f5af8bfdad63b9f72a,"[""208.80.154.10""]",f,"[""server:HAProxy""]",TLSv1.3,2a315459e463d249065f83e3fdf83c6083add95422a202f5af8bfdad63b9f72a,clean,NaN,informational,2026-11-28T08:54:12.971051+00:00,2026-11-28T08:54:12.971051+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,a882c1838c2d3124e4b0,tenant_wikimedia,wikitech-static-ord.wikimedia.org,fqdn:wikitech-static-ord.wikimedia.org,fqdn:wikitech-static-ord.wikimedia.org,[],f,[],NaN,NaN,clean,NaN,informational,2026-11-28T08:54:56.420576+00:00,2026-11-28T08:54:56.420576+00:00
139,2f927c2057208ba5d64f,tenant_wikimedia,wikitech-static.wikimedia.org,cert:770fce4bb078d2a220b0d7b6072ad092592f349ef35fb17cf17b0ed21962ec40,cert:770fce4bb078d2a220b0d7b6072ad092592f349ef35fb17cf17b0ed21962ec40,"[""3.126.27.158""]",f,"[""server:nginx""]",TLSv1.3,770fce4bb078d2a220b0d7b6072ad092592f349ef35fb17cf17b0ed21962ec40,clean,NaN,informational,2026-11-28T08:54:57.394474+00:00,2026-11-28T08:54:57.394474+00:00
140,105110b9f2b5d268ce4b,tenant_wikimedia,wikitech.wikimedia.org,cert:14a7accb60f49ba0bedc0986c99750b37d25f0151312eb9d8f1a8a3f8748426f,fqdn:dyna.wikimedia.org,"[""185.15.59.224""]",f,"[""server:HAProxy""]",TLSv1.3,14a7accb60f49ba0bedc0986c99750b37d25f0151312eb9d8f1a8a3f8748426f,clean,NaN,informational,2026-11-28T08:54:56.943419+00:00,2026-11-28T08:54:56.943419+00:00
141,b8f3e7b9b254efb05fc5,tenant_wikimedia,www.policy.wikimedia.org,fqdn:www.policy.wikimedia.org,fqdn:www.policy.wikimedia.org,[],f,[],NaN,NaN,clean,NaN,informational,2026-11-28T08:54:56.635164+00:00,2026-11-28T08:54:56.635164+00:00


In [11]:
sql(f"SELECT * FROM gold_asset_findings WHERE tenant_id = '{TENANT}'")

,asset_id,tenant_id,source,exposure_type,validation_tier,detected_at


In [12]:
sql(f"SELECT * FROM gold_asset_edges WHERE tenant_id = '{TENANT}'")

,edge_id,tenant_id,src_node,relation,dst_node,is_shared_infra,observed_at
0,101f41d0932b1b68c2dc,tenant_wikimedia,fqdn:planet.wikimedia.org,RESOLVES_TO,ip:185.15.59.224,f,2026-11-28T08:54:44.060104+00:00
1,18d244705884db411fbc,tenant_wikimedia,fqdn:wikitech.wikimedia.org,CNAME_TO,fqdn:dyna.wikimedia.org,f,2026-11-28T08:54:56.530151+00:00
2,18e149ade53a0b171f2f,tenant_wikimedia,cert:5f31019759aeb6cbc70f6fc566a98d42e56d414a219d070455c4aa461a2d1a2d,COVERS_SAN,fqdn:frmon.wikimedia.org,f,2026-11-28T08:54:25.559534+00:00
3,2423af96bb6a75b4ae7f,tenant_wikimedia,fqdn:gerrit.wikimedia.org,RESOLVES_TO,ip:185.15.59.225,f,2026-11-28T08:54:24.480236+00:00
4,24de505ec3ff3a89b820,tenant_wikimedia,cert:c17ef8655c43b8dc2ba9b52f775714e48fdf7da81c44994f331abbe664eca88b,COVERS_SAN,fqdn:fundraising.frdev.wikimedia.org,f,2026-11-28T08:54:25.416736+00:00
...,...,...,...,...,...,...,...
244,d3b1e3841c243209f66f,tenant_wikimedia,fqdn:svn.wikimedia.org,RESOLVES_TO,ip:185.15.59.224,f,2026-11-28T08:54:46.621237+00:00
245,d9d4c8954d25913ddaee,tenant_wikimedia,fqdn:alerts.wikimedia.org,RESOLVES_TO,ip:208.80.154.78,f,2026-11-28T08:54:09.060837+00:00
246,de1789aa4760498a74ee,tenant_wikimedia,fqdn:payments-staging.wikimedia.org,RESOLVES_TO,ip:208.80.155.10,f,2026-11-28T08:54:43.761716+00:00
247,dfcc264eb7e3e1894d90,tenant_wikimedia,fqdn:civicrm.wikimedia.org,RESOLVES_TO,ip:208.80.155.17,f,2026-11-28T08:54:11.933451+00:00


In [13]:
# Row counts for all tables, including staging
tables = sql("SELECT tablename FROM pg_tables WHERE schemaname='public' ORDER BY 1")["tablename"]
pd.Series({t: int(sql(f"SELECT count(*) AS n FROM {t}")["n"][0]) for t in tables}, name="rows")

gold_asset_clusters_staging    143
gold_asset_edges               330
gold_asset_edges_staging       249
gold_asset_findings              6
gold_asset_findings_staging      0
gold_asset_sources_staging       1
gold_assets                    172
gold_assets_staging              1
Name: rows, dtype: int64

## 6. Serving API (clusters)

In [14]:
script = f"import urllib.request as u;print(u.urlopen('http://localhost:8080/tenants/{TENANT}/clusters',timeout=30).read().decode())"
print(json.dumps(json.loads(kubectl("exec", "deploy/asset-api", "--", "python", "-c", script)), indent=2))

[]
